<a href="https://colab.research.google.com/github/shomere/formative_2group3_ml_techs1_swac/blob/main/bilstm_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BiLSTM — Swahili Audio Classification

Loads pre-extracted MFCC+delta features from `w1.5_energy_mfcc40_d/`.

- Input shape: `(N, 150, 120)` — 150 time-steps × 120 features (40 MFCCs + Δ + ΔΔ)
- 12-class spoken-word classification
- Evaluated with accuracy + macro-F1

**Why BiLSTM for this task?**  
Speech is a temporal sequence where phonetic context flows both forward and backward. A BiLSTM processes each MFCC frame in both directions, capturing coarticulation effects that a unidirectional LSTM misses. This is well-supported in the speech recognition literature (Graves & Schmidhuber, 2005; Graves et al., 2013).

In [1]:
# ── 0. Environment setup ──────────────────────────────────────────────────────
import os
from pathlib import Path

USE_COLAB = 'COLAB_GPU' in os.environ

if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Upload the w1.5_energy_mfcc40_d folder to your Drive root, then set:
    FEAT_DIR = '/content/drive/MyDrive/w1.5_energy_mfcc40_d'
    LOG_DIR  = '/content/drive/MyDrive/reports'
    FIG_DIR  = '/content/drive/MyDrive/reports/figures'
else:
    # Local — paths relative to repo root
    FEAT_DIR = '../w1.5_energy_mfcc40_d'
    LOG_DIR  = '../reports'
    FIG_DIR  = '../reports/figures'

Path(FIG_DIR).mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

print('Feature dir:', FEAT_DIR)
print('Files:', os.listdir(FEAT_DIR))

Mounted at /content/drive
Feature dir: /content/drive/MyDrive/w1.5_energy_mfcc40_d
Files: ['labels.json', 'val.npz', 'train.npz', 'test.npz', 'w1.5_energy_mfcc40_d']


In [2]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

print('TF version :', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

TF version : 2.20.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ── 2. Load features ──────────────────────────────────────────────────────────
fd = Path(FEAT_DIR)

tr = np.load(fd / 'train.npz', allow_pickle=True)
va = np.load(fd / 'val.npz',   allow_pickle=True)
te = np.load(fd / 'test.npz',  allow_pickle=True)

X_tr, y_tr = tr['X'].astype(np.float32), tr['y']
X_va, y_va = va['X'].astype(np.float32), va['y']
X_te, y_te = te['X'].astype(np.float32), te['y']

label2id   = json.loads((fd / 'labels.json').read_text())
id2label   = {v: k for k, v in label2id.items()}
CLASS_NAMES = [id2label[i] for i in range(len(id2label))]
N_CLASSES   = len(CLASS_NAMES)

print(f'Train : X{X_tr.shape}  y{y_tr.shape}')
print(f'Val   : X{X_va.shape}  y{y_va.shape}')
print(f'Test  : X{X_te.shape}  y{y_te.shape}')
print(f'Classes ({N_CLASSES}): {CLASS_NAMES}')

Train : X(2940, 151, 120)  y(2940,)
Val   : X(630, 151, 120)  y(630,)
Test  : X(630, 151, 120)  y(630,)
Classes (12): ['hapana', 'kumi', 'mbili', 'moja', 'nane', 'ndio', 'nne', 'saba', 'sita', 'tano', 'tatu', 'tisa']


In [4]:
# ── 3. Normalise (per-feature z-score, fitted on train only) ──────────────────
# Shape: (1, 1, 120) — broadcast over samples and time-steps
mu  = X_tr.mean(axis=(0, 1), keepdims=True)
std = X_tr.std(axis=(0, 1),  keepdims=True) + 1e-8

X_tr = (X_tr - mu) / std
X_va = (X_va - mu) / std
X_te = (X_te - mu) / std

T, F = X_tr.shape[1], X_tr.shape[2]   # 150 timesteps, 120 features
print(f'Normalised. Timesteps={T}, Features={F}')

Normalised. Timesteps=151, Features=120


In [5]:
# ── 4. Build BiLSTM model ─────────────────────────────────────────────────────
# Architecture rationale:
#   - Two BiLSTM layers: first returns sequences (captures local temporal
#     patterns), second returns a single context vector (global summary).
#   - BatchNorm stabilises training; Dropout prevents co-adaptation.
#   - Dense(128) projects to a compact representation before softmax.

def build_bilstm(timesteps, features, n_classes,
                 units=128, dropout=0.3):
    inp = keras.Input(shape=(timesteps, features), name='mfcc_input')

    x = layers.Bidirectional(
            layers.LSTM(units, return_sequences=True),
            name='bilstm_1')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Bidirectional(
            layers.LSTM(units // 2, return_sequences=False),
            name='bilstm_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)

    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(dropout / 2)(x)
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    return keras.Model(inp, out, name='BiLSTM')


model = build_bilstm(T, F, N_CLASSES)
model.summary()

Model: "BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mfcc_input (InputLayer)         │ (None, 151, 120)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_1 (Bidirectional)        │ (None, 151, 256)       │       254,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 151, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 151, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_2 (Bidirectional)        │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 12)             │         1,548 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 438,924 (1.67 MB)

 Trainable params: 438,156 (1.67 MB)

 Non-trainable params: 768 (3.00 KB)

In [6]:
# ── 5. Compile & callbacks ────────────────────────────────────────────────────
LR       = 1e-3
EPOCHS   = 60
BATCH    = 64
PATIENCE = 12

model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=PATIENCE,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=6,
        min_lr=1e-6, verbose=1)
]

print(f'Training config: lr={LR}, epochs={EPOCHS}, batch={BATCH}, patience={PATIENCE}')

Training config: lr=0.001, epochs=60, batch=64, patience=12


In [7]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
t0 = time.time()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_va, y_va),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1
)

train_seconds = round(time.time() - t0, 1)
print(f'\nTraining time: {train_seconds}s')

Epoch 1/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - accuracy: 0.0976 - loss: 2.8736 - val_accuracy: 0.1270 - val_loss: 2.4589 - learning_rate: 0.0010
Epoch 2/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.2139 - loss: 2.2980 - val_accuracy: 0.1444 - val_loss: 2.4189 - learning_rate: 0.0010
Epoch 3/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.3207 - loss: 2.0004 - val_accuracy: 0.1556 - val_loss: 2.3764 - learning_rate: 0.0010
Epoch 4/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.4211 - loss: 1.7456 - val_accuracy: 0.1857 - val_loss: 2.3295 - learning_rate: 0.0010
Epoch 5/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5310 - loss: 1.4648 - val_accuracy: 0.1968 - val_loss: 2.3053 - learning_rate: 0.0010
Epoch 6/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.6102 - loss: 1.2063 - val_accuracy: 0.2222 - val_loss: 2.2863 - learning_rate: 0.0010
Epoch 7/60
46/46 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.6932 - loss: 0.9577 - val_ac

In [8]:
# ── 7. Learning curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('BiLSTM — Learning Curves', fontsize=13)
plt.tight_layout()

fig_path = Path(FIG_DIR) / 'bilstm_learning_curves.png'
plt.savefig(fig_path, dpi=150)
plt.show()
print('Saved:', fig_path)

Saved: /content/drive/MyDrive/reports/figures/bilstm_learning_curves.png


In [9]:
# ── 8. Validation evaluation ──────────────────────────────────────────────────
y_va_pred = model.predict(X_va, verbose=0).argmax(axis=1)
val_acc   = accuracy_score(y_va, y_va_pred)
val_f1    = f1_score(y_va, y_va_pred, average='macro')

print(f'Val  accuracy={val_acc:.4f}  macro-F1={val_f1:.4f}')

Val  accuracy=0.3762  macro-F1=0.3751


In [10]:
# ── 9. Test evaluation ────────────────────────────────────────────────────────
t1 = time.time()
y_te_pred = model.predict(X_te, verbose=0).argmax(axis=1)
infer_s   = round(time.time() - t1, 2)

test_acc = accuracy_score(y_te, y_te_pred)
test_f1  = f1_score(y_te, y_te_pred, average='macro')

print(f'Test accuracy={test_acc:.4f}  macro-F1={test_f1:.4f}')
print(f'Inference time: {infer_s}s\n')
print(classification_report(y_te, y_te_pred, target_names=CLASS_NAMES, digits=3))

Test accuracy=0.3984  macro-F1=0.3939
Inference time: 0.37s

              precision    recall  f1-score   support

      hapana      0.276     0.308     0.291        52
        kumi      0.405     0.321     0.358        53
       mbili      0.378     0.327     0.351        52
        moja      0.333     0.340     0.336        53
        nane      0.453     0.453     0.453        53
        ndio      0.439     0.346     0.387        52
         nne      0.448     0.491     0.468        53
        saba      0.585     0.585     0.585        53
        sita      0.544     0.712     0.617        52
        tano      0.140     0.115     0.126        52
        tatu      0.226     0.269     0.246        52
        tisa      0.509     0.509     0.509        53

    accuracy                          0.398       630
   macro avg      0.395     0.398     0.394       630
weighted avg      0.395     0.398     0.394       630



In [11]:
# ── 10. Confusion matrix ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_te_pred,
    display_labels=CLASS_NAMES,
    xticks_rotation=45,
    ax=ax, colorbar=False
)
ax.set_title('BiLSTM — Test Confusion Matrix')
plt.tight_layout()

cm_path = Path(FIG_DIR) / 'cm_bilstm.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print('Saved:', cm_path)

Saved: /content/drive/MyDrive/reports/figures/cm_bilstm.png


In [12]:
# ── 11. Log to experiments.csv ────────────────────────────────────────────────
row = {
    'date':              datetime.now().strftime('%Y-%m-%d %H:%M'),
    'model':             'bilstm',
    'features':          'w1.5_energy_mfcc40_d',
    'params':            json.dumps({'units': 128, 'dropout': 0.3,
                                     'lr': LR, 'batch': BATCH}),
    'val_acc':           round(val_acc,  4),
    'val_macro_f1':      round(val_f1,   4),
    'test_acc':          round(test_acc, 4),
    'test_macro_f1':     round(test_f1,  4),
    'train_seconds':     train_seconds,
    'inference_seconds': infer_s,
}

log = Path(LOG_DIR) / 'experiments.csv'
pd.DataFrame([row]).to_csv(log, mode='a', header=not log.exists(), index=False)
print('Logged to', log)
print(row)

Logged to /content/drive/MyDrive/reports/experiments.csv
{'date': '2026-10-04 07:55', 'model': 'bilstm', 'features': 'w1.5_energy_mfcc40_d', 'params': '{"units": 128, "dropout": 0.3, "lr": 0.001, "batch": 64}', 'val_acc': 0.3762, 'val_macro_f1': 0.3751, 'test_acc': 0.3984, 'test_macro_f1': 0.3939, 'train_seconds': 96.0, 'inference_seconds': 0.37}


In [13]:
# ── 12. Error analysis — top confused pairs ───────────────────────────────────
cm = confusion_matrix(y_te, y_te_pred)
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)

print('Top-5 confused pairs (true → predicted, count):')
top5 = np.argsort(cm_off.flatten())[::-1][:5]
for idx in top5:
    r, c = divmod(int(idx), N_CLASSES)
    print(f'  {CLASS_NAMES[r]:8s} → {CLASS_NAMES[c]:8s}  ({cm_off[r, c]} errors)')

print()
print('Per-class accuracy:')
for i, name in enumerate(CLASS_NAMES):
    total   = cm[i].sum()
    correct = cm[i, i]
    print(f'  {name:8s}: {correct}/{total} = {correct/total:.2%}')

Top-5 confused pairs (true → predicted, count):
  tisa     → sita      (12 errors)
  tano     → tatu      (12 errors)
  hapana   → tatu      (12 errors)
  moja     → nane      (9 errors)
  tano     → hapana    (8 errors)

Per-class accuracy:
  hapana  : 16/52 = 30.77%
  kumi    : 17/53 = 32.08%
  mbili   : 17/52 = 32.69%
  moja    : 18/53 = 33.96%
  nane    : 24/53 = 45.28%
  ndio    : 18/52 = 34.62%
  nne     : 26/53 = 49.06%
  saba    : 31/53 = 58.49%
  sita    : 37/52 = 71.15%
  tano    : 6/52 = 11.54%
  tatu    : 14/52 = 26.92%
  tisa    : 27/53 = 50.94%


In [14]:
# ── 13. Comparison with baselines ─────────────────────────────────────────────
baselines = {
    'LogReg (best baseline)': {'val_f1': 0.7155, 'test_f1': 0.7083},
    'SVM':                    {'val_f1': 0.7046, 'test_f1': 0.7027},
    'Random Forest':          {'val_f1': 0.6847, 'test_f1': 0.6452},
}

print('Model comparison (macro-F1):')
print(f'{"Model":<28} {"Val F1":>8} {"Test F1":>9}')
print('-' * 48)
for name, scores in baselines.items():
    print(f'{name:<28} {scores["val_f1"]:>8.4f} {scores["test_f1"]:>9.4f}')
print(f'{"BiLSTM (this run)":<28} {val_f1:>8.4f} {test_f1:>9.4f}')

delta = test_f1 - 0.7083
print(f'\nBiLSTM vs best baseline: {delta:+.4f} macro-F1 on test')

Model comparison (macro-F1):
Model                          Val F1   Test F1
------------------------------------------------
LogReg (best baseline)         0.7155    0.7083
SVM                            0.7046    0.7027
Random Forest                  0.6847    0.6452
BiLSTM (this run)              0.3751    0.3939

BiLSTM vs best baseline: -0.3144 macro-F1 on test


In [15]:
# ── 14. Save model ────────────────────────────────────────────────────────────
model_dir = Path(LOG_DIR).parent / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'bilstm_swahili.keras'
model.save(model_path)
print('Model saved to', model_path)

Model saved to /content/drive/MyDrive/models/bilstm_swahili.keras
